# 01 — Baseline: Full Fine-Tune vs LoRA on Banking77

Part of [bert-lora-finetuning](../README.md). Run this notebook **first** — its output feeds the
`FULL_FINETUNE_ACCURACY` reference value used in `02_lora_rank_sweep.ipynb`.

**Task:** 77-class intent classification on [Banking77](https://huggingface.co/datasets/PolyAI/banking77)
(bank customer-support queries).

**Compares:** `bert-base-uncased` fully fine-tuned vs. the same model fine-tuned with LoRA (`r=8`,
via HuggingFace `peft`), on identical data and training config.

**Run on Colab or Kaggle with a GPU runtime.** Full fine-tuning 77-class BERT on CPU is impractically slow.


## Setup

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate torch scikit-learn


In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected — switch to a GPU runtime before running the training cells.")

RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)


## 1. Data

Banking77: ~10k train / ~3k test bank customer-support queries, 77 intent classes. Same style of problem
as the Desible intent classifier (multi-class intent from short text), on a public benchmark.


In [ ]:
dataset = load_dataset("PolyAI/banking77")
print(dataset)

label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"Number of intent classes: {num_labels}")
print("First 10 labels:", label_names[:10])
print("Example row:", dataset["train"][0])


In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_ds = tokenized["train"]
eval_ds = tokenized["test"]
print(f"Train size: {len(train_ds)}, Test size: {len(eval_ds)}")

# padding="max_length" above means every batch is already fixed-size, so Trainer doesn't need a
# tokenizer/data collator passed in for dynamic padding — sidesteps the tokenizer= vs
# processing_class= rename across transformers versions.


## 2. Shared helpers

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1}

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def peak_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


## 3. Full fine-tuning (baseline)

In [ ]:
full_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
full_model.to(device)

full_trainable, full_total = count_trainable_params(full_model)
print(f"[Full fine-tune] Trainable params: {full_trainable:,} / {full_total:,} "
      f"({100 * full_trainable / full_total:.2f}%)")

full_args = TrainingArguments(
    output_dir="./full_finetune_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

full_trainer = Trainer(
    model=full_model, args=full_args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

reset_peak_memory()
start = time.time()
full_trainer.train()
full_train_time = time.time() - start
full_peak_mem = peak_memory_mb()

print(f"[Full fine-tune] Training time: {full_train_time:.1f}s")
if full_peak_mem:
    print(f"[Full fine-tune] Peak GPU memory: {full_peak_mem:.1f} MB")


In [ ]:
full_eval_results = full_trainer.evaluate()
print("[Full fine-tune] Eval results:", full_eval_results)


## 4. LoRA fine-tuning (via HuggingFace `peft`, r=8)

In [ ]:
lora_base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,           # scaling factor applied is alpha/r = 2
    lora_dropout=0.1,
    target_modules=["query", "value"],
)

lora_model = get_peft_model(lora_base_model, lora_config)
lora_model.to(device)
lora_model.print_trainable_parameters()

lora_trainable, lora_total = count_trainable_params(lora_model)


In [ ]:
lora_args = TrainingArguments(
    output_dir="./lora_out",
    learning_rate=2e-4,      # LoRA typically wants a higher LR than full fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

lora_trainer = Trainer(
    model=lora_model, args=lora_args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

reset_peak_memory()
start = time.time()
lora_trainer.train()
lora_train_time = time.time() - start
lora_peak_mem = peak_memory_mb()

print(f"[LoRA] Training time: {lora_train_time:.1f}s")
if lora_peak_mem:
    print(f"[LoRA] Peak GPU memory: {lora_peak_mem:.1f} MB")


In [ ]:
lora_eval_results = lora_trainer.evaluate()
print("[LoRA] Eval results:", lora_eval_results)


## 5. Comparison

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "full_finetune",
        "trainable_params": full_trainable,
        "trainable_pct": round(100 * full_trainable / full_total, 3),
        "accuracy": full_eval_results["eval_accuracy"],
        "macro_f1": full_eval_results["eval_macro_f1"],
        "train_time_s": round(full_train_time, 1),
        "peak_mem_mb": round(full_peak_mem, 1) if full_peak_mem else None,
    },
    {
        "method": "lora_r8",
        "trainable_params": lora_trainable,
        "trainable_pct": round(100 * lora_trainable / lora_total, 3),
        "accuracy": lora_eval_results["eval_accuracy"],
        "macro_f1": lora_eval_results["eval_macro_f1"],
        "train_time_s": round(lora_train_time, 1),
        "peak_mem_mb": round(lora_peak_mem, 1) if lora_peak_mem else None,
    },
])
comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["#4C72B0", "#55A868"]

axes[0].bar(comparison["method"], comparison["trainable_params"], color=colors)
axes[0].set_title("Trainable Parameters (log scale)")
axes[0].set_yscale("log")

axes[1].bar(comparison["method"], comparison["accuracy"], color=colors)
axes[1].set_title("Accuracy")
axes[1].set_ylim(0, 1)

axes[2].bar(comparison["method"], comparison["train_time_s"], color=colors)
axes[2].set_title("Training Time (s)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_baseline_comparison.png", dpi=150)
plt.show()


In [ ]:
# Save results so 02_lora_rank_sweep.ipynb and the README can pull real numbers instead of placeholders.
comparison.to_csv(f"{RESULTS_DIR}/01_baseline_comparison.csv", index=False)
with open(f"{RESULTS_DIR}/01_baseline_comparison.json", "w") as f:
    json.dump(comparison.to_dict(orient="records"), f, indent=2)

print(f"Saved results to {RESULTS_DIR}/01_baseline_comparison.{{csv,json}}")
print()
print("Copy this into 02_lora_rank_sweep.ipynb's FULL_FINETUNE_ACCURACY constant:")
print(full_eval_results["eval_accuracy"])


**Write this down, don't skip it:** how much accuracy did LoRA recover, for what fraction of the
trainable parameters, training time, and memory? That trade-off — not either number alone — is the
actual finding. Put it in the README's results table once you have it.
